Lesson is below this is my follow along


Things to remmember \
OC_STORM,  Replace attack detection with the new keyboard input instead of mouse\

In [ ]:
import mss
import cv2
import pywinctl
import numpy as np
import time
from pynput import mouse, keyboard
import pydirectinput
from pathlib import Path

IMG_DIR = r'C:\Users\Owner\Desktop\PythonStuff\ObsidianVault\images'
TARGET_WINDOW = "Hollow Knight"
TOTAL_MASKS = 9
LAST_CHECKED_MASK = 0
MATCH_SCALE = 0.5
MATCH_THRESHOLD = 0.55


sct = mss.MSS()
windows = pywinctl.getWindowsWithTitle(TARGET_WINDOW)

if not windows:
    print("No Target Window Found!")
    exit()
    
win = windows[0]

lower_mask_pink = np.array([210, 205, 222])  # B, G, R
upper_mask_pink = np.array([230, 222, 240])  # B, G, R

def GetMasks(health_frame):
    frame_height, frame_width, _ = health_frame.shape
    mask_width = health_frame.shape[1] // TOTAL_MASKS
    
    current_masks = 0
    for i in range(TOTAL_MASKS):
        start_x = i * mask_width
        end_x = (i + 1) * mask_width
        
        single_mask_box = health_frame[0:frame_height, start_x:end_x]
        
        isolated_color_box = cv2.inRange(single_mask_box, lower_mask_pink, upper_mask_pink)
        
        box_match_pixels = np.sum(isolated_color_box == 255)
        
        if box_match_pixels > 30: 
            current_masks += 1
    return current_masks

health_debounce, hp_debounce_frames = False, 10
current_hp_deb_frame = 0

def Detections(health_frame, processed_frame):
    global LAST_CHECKED_MASK, current_hp_deb_frame, hp_debounce_frames, health_debounce
    brightness = np.mean(processed_frame)
    
    # Health down debounce
    if current_hp_deb_frame < hp_debounce_frames and not health_debounce:
        current_hp_deb_frame += 1
    elif current_hp_deb_frame >= hp_debounce_frames:
        current_hp_deb_frame = 0
        health_debounce = True
    
    current_masks = GetMasks(health_frame=health_frame)
    
    if LAST_CHECKED_MASK < current_masks: # edge case fixes
        LAST_CHECKED_MASK = current_masks
    
    # checks if health went down
    if LAST_CHECKED_MASK > current_masks and brightness < 235 and health_debounce:
        amount_down = LAST_CHECKED_MASK - current_masks
        
        if amount_down > 2: # edge case for teleports
            return
        print("Masks:", current_masks)
        LAST_CHECKED_MASK = current_masks
        
    
    # checks if player died
    if current_masks <= 0 and brightness < 230:
        print("Player Died!")
        time.sleep(5)
        LAST_CHECKED_MASK = GetMasks(health_frame=health_frame)
    if brightness > 250:
        print("In Loading!")
        time.sleep(3)
        

def CheckBossHit(time_lost_boss):
    if time_lost_boss > 0.20 and time_lost_boss < 1.00 and (time.time() -last_attack_time) < 1:
        print("Hit Boss!")

#-Hornet Colors-
# Just need red dress
HORNET_RED_LO_1 = np.array([170, 90, 70])  
HORNET_RED_HI_1 = np.array([180, 255, 220]) 

time_lost_boss = 0.0
lost_boss = False
last_seen_boss = time.time()
last_time_stamp = time.time()

def GetBossBounds(raw_frame):
    global lost_boss, last_seen_boss, time_lost_boss, last_time_stamp
    
    hsv_frame = cv2.cvtColor(raw_frame, cv2.COLOR_BGR2HSV)
  
    mask = cv2.inRange(hsv_frame, HORNET_RED_LO_1, HORNET_RED_HI_1)
    
    open_kernel = np.ones((3, 3), np.uint8)
    clean_mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, open_kernel)
    
    close_kernel = np.ones((9, 9), np.uint8)
    final_mask = cv2.morphologyEx(clean_mask, cv2.MORPH_CLOSE, close_kernel)
    
    contours, _ = cv2.findContours(final_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    #cv2.imshow("Mask", final_mask)
    
    if not contours:
        if not lost_boss:
            last_time_stamp = time.time()
        lost_boss = True
        return None

    valid_boss_parts = [cnt for cnt in contours if cv2.contourArea(cnt) > 150]
    
    if not valid_boss_parts:
        if not lost_boss:
            last_time_stamp = time.time()
        lost_boss = True
        return None
    
    # used to detect if a hit occured
    if lost_boss:
        time_lost_boss = time.time() - last_time_stamp
        #print(f"Boss was lost for {time_lost_boss:.2f} seconds")
        CheckBossHit(time_lost_boss=time_lost_boss)
        
        lost_boss = False
        last_seen_boss = time.time()

    
    
    all_points = np.vstack(valid_boss_parts)
    x, y, w, h = cv2.boundingRect(all_points)
    
    # Draw the newly stabilized master box
    cv2.rectangle(raw_frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
    
    return (x, y, w, h)

def GetPlayerImages(flip=False, scale=MATCH_SCALE):
    folder = Path(r"C:\Users\Owner\Desktop\PythonStuff\ObsidianVault\images\hollowknight\player")
    images = []
    valid_extensions = {'.png', '.jpg', '.jpeg', '.bmp'}

    for file_path in folder.rglob("*"):
        if file_path.is_file() and file_path.suffix.lower() in valid_extensions:
            img = cv2.imread(str(file_path), cv2.IMREAD_GRAYSCALE)
            if img is not None:
                # Turn the template into a clean binary mask (White head, black eyes)
                _, thresh_t = cv2.threshold(img, 220, 255, cv2.THRESH_BINARY)
                
                variants = [thresh_t]
                if flip:
                    variants.append(cv2.flip(thresh_t, 1))

                for v in variants:
                    h, w = v.shape
                    new_w, new_h = max(1, int(w * scale)), max(1, int(h * scale))
                    small = cv2.resize(v, (new_w, new_h), interpolation=cv2.INTER_AREA)
                    
                    # Clean up any blur from resizing to keep it strictly binary
                    _, small_binary = cv2.threshold(small, 127, 255, cv2.THRESH_BINARY)
                    images.append(small_binary)
    return images
    
player_templates = GetPlayerImages(flip=True)

PLAYER_WHITE_LO = np.array([0, 0, 240])    
PLAYER_WHITE_HI = np.array([180, 15, 255]) 

def Get_PlayerBounds(raw_frame, boss_bounds=None):
    global player_templates
    
    # 1. Convert to HSV and isolate pure white pixels instantly
    hsv_frame = cv2.cvtColor(raw_frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv_frame, PLAYER_WHITE_LO, PLAYER_WHITE_HI)
    
    # 2. Resize the white mask down to your processing scale
    h, w = mask.shape
    small_w, small_h = max(1, int(w * MATCH_SCALE)), max(1, int(h * MATCH_SCALE))
    mask_small = cv2.resize(mask, (small_w, small_h), interpolation=cv2.INTER_AREA)
    _, mask_small = cv2.threshold(mask_small, 127, 255, cv2.THRESH_BINARY)
    
    # 3. Blank out Hornet and the HUD on the mask layer
    if boss_bounds is not None:
        bx, by, bw, bh = boss_bounds
        sbx, sby = int(bx * MATCH_SCALE), int(by * MATCH_SCALE)
        sbw, sbh = int(bw * MATCH_SCALE), int(bh * MATCH_SCALE)
        padding = int(15 * MATCH_SCALE + 25)
        
        cv2.rectangle(
            mask_small, 
            (max(0, sbx - padding), max(0, sby - padding)), 
            (min(mask_small.shape[1], sbx + sbw + padding), min(mask_small.shape[0], sby + sbh + padding)), 
            0, # Erase with black
            -1
        )
    
    # Blank out HUD
    cv2.rectangle(mask_small, (0, 0), (int(500 * MATCH_SCALE), int(200 * MATCH_SCALE)), 0, -1)
    # cv2.imshow("AI Vision Window", mask_small)

    best_max_val = -1
    best_max_loc = None
    best_w, best_h = 0, 0
    
    # 4. Slide our binary head templates over the binary frame mask
    for idx, template in enumerate(player_templates):
        th, tw = template.shape 
        
        match_map = cv2.matchTemplate(mask_small, template, cv2.TM_CCOEFF_NORMED)
        _, max_val, _, max_loc = cv2.minMaxLoc(match_map)
        
        if max_val > best_max_val:
            best_max_val = max_val
            best_max_loc = max_loc
            best_w, best_h = tw, th

    # 5. Evaluate confidence score
    if best_max_val > MATCH_THRESHOLD:
        inv_scale = 1.0 / MATCH_SCALE
        hx = int(best_max_loc[0] * inv_scale)
        hy = int(best_max_loc[1] * inv_scale)
        tw = int(best_w * inv_scale)
        th = int(best_h * inv_scale)
        
        px = hx - 2
        py = hy
        pw = tw + 4
        ph = int(th * 1.5)
        
        ph = min(ph, raw_frame.shape[0] - py)
        
        # Draw Blue Tracking Box
        cv2.rectangle(raw_frame, (px, py), (px + pw, py + ph), (255, 0, 0), 2)
        
        return (px, py, pw, ph)
        
    return None
    
last_attack_time = 0.0
macro_deb = False

def on_click(x, y, button, pressed):
    global last_attack_time
    if button == mouse.Button.left and pressed:
        last_attack_time = time.time()

def on_keypress(key):
    global macro_deb
    if key == keyboard.KeyCode.from_char("n") and macro_deb == False:
        macro_deb = True
        print("starting macro")
        reset_game()
        time.sleep(1)
        macro_deb = False

# listeners
mouse_listener = mouse.Listener(on_click=on_click)
key_listener = keyboard.Listener(on_press=on_keypress)

key_listener.start()
mouse_listener.start()

# --- THE MODEL ---
pydirectinput.PAUSE = 0.1

def take_action(self, action):
    if action != 1:
        pydirectinput.keyUp('a')
    if action != 2:
        pydirectinput.keyUp('d')

    # 2. Execute the new action
    if action == 0:  # Do nothing
        pass
        
    elif action == 1:  # Move Left
        pydirectinput.keyDown('a')
        
    elif action == 2:  # Move Right
        pydirectinput.keyDown('d')
        
    elif action == 3:  # Attack
        pydirectinput.press('r') 
        
    elif action == 4:  # Jump
        pydirectinput.press('space')
        
    elif action == 5:  # Dash
        pydirectinput.press('t') 

def reset_game():
    pydirectinput.press('space')
    time.sleep(1)
    pydirectinput.press('w')
    time.sleep(0.3)
    pydirectinput.press('space')
    time.sleep(3)
    loaded_in()
    return


def screen_capture():
    global LAST_CHECKED_MASK
    
    window_dims = {
        "left": win.left,
        "top": win.top,
        "width": win.width,
        "height": win.height
    }
    raw_frame = np.array(sct.grab(window_dims))
    
    health_frame = raw_frame[80:120, 245:675, :3] # works when full screen
     
    masks = GetMasks(health_frame=health_frame)
    
    if masks > 0 and LAST_CHECKED_MASK == 0:
        LAST_CHECKED_MASK = masks

    processed_frame = cv2.resize(raw_frame, (128, 90), interpolation=cv2.INTER_LINEAR)
    processed_frame = cv2.cvtColor(processed_frame, cv2.COLOR_BGRA2GRAY)
    
    
    boss_box = GetBossBounds(raw_frame=raw_frame)
    player_box = Get_PlayerBounds(raw_frame=raw_frame, boss_bounds=boss_box)
    
    #cv2.imshow("Raw - altered", raw_frame)
    cv2.imshow("Display", processed_frame)
    cv2.imshow("HealthBar", health_frame)

    # print(f"HEALTH MASKS DETECTED: {masks} / {TOTAL_MASKS} {LAST_CHECKED_MASK}", end="\r")
    Detections(health_frame=health_frame, processed_frame=processed_frame)
    return processed_frame
    
def loaded_in():
    return

while True:
    processed_frame = screen_capture()
    
    if cv2.waitKey(1) & 0xFF == ord("q"):
        cv2.destroyAllWindows()
        key_listener.stop()
        mouse_listener.stop()
        break

starting macro
In Boss Arena!
starting macro
In Boss Arena!
starting macro
In Boss Arena!
starting macro
In Boss Arena!
starting macro
In Boss Arena!
starting macro
In Boss Arena!


<div align="center">

# Masterclass Curriculum: Engineering a 2D Adversarial Boss-Slayer AI
## TensorFlow Edition

A polished notebook version of the original blueprint for vision capture, telemetry, action mapping, and DQN training in a 2D boss-fight arena.

</div>

> **Notebook intent:** present the system design clearly, preserve the key engineering details, and make the curriculum easier to scan during study or implementation.

---

## At a Glance

| Section | What it covers |
| --- | --- |
| Step 1 | Frame capture, grayscale compression, and 4-frame temporal stacking |
| Step 2 | Health-mask telemetry, boss-hit signals, and reward shaping |
| Step 3 | Action-space mapping and controller unlatching |
| Step 4 | CNN-based DQN design and Bellman optimization |
| Step 5 | The full runtime loop that binds perception, actuation, and learning |

### System Overview

```text
+------------------+     Raw Pixels     +---------------------------+
|   Game Window    | -----------------> |  Step 1: Vision Pipeline  |
+------------------+                    |  Capture & Frame Stack    |
         ^                              +---------------------------+
         | Virtual Inputs                           |
+------------------+                    +---------------------------+
| Step 3: Actuator |                    | Step 2: Telemetry Engine  |
|   vgamepad       |                    |  Mask & Hit Detection     |
+------------------+                    +---------------------------+
         ^                              +---------------------------+
         | Action Selection                      |
         +---------------------------------- State & Reward Tensors
                                                  |
                                                  v
                                        +---------------------------+
                                        |   Step 4: Policy Graph   |
                                        |      DQN + GradientTape  |
                                        +---------------------------+
```

The key design idea is simple: the model does not just see a frame, it sees motion, danger, and feedback as a structured state tensor.

---

## Master API Dictionary

| Area | Core tools | Why they matter |
| --- | --- | --- |
| Window capture | `pygetwindow.getWindowsWithTitle()`, `mss.mss()`, `sct.grab()` | Locates the game and pulls pixels directly from the screen buffer |
| Vision processing | `cv2.cvtColor()`, `cv2.resize()`, `np.array()` | Converts raw frames into compact grayscale tensors |
| Virtual input | `vgamepad.VX360Gamepad()`, `press_button()`, `release_button()`, `update()` | Turns action indices into real controller events |
| Learning graph | `tf.keras.layers.Conv2D`, `tf.GradientTape()`, `optimizer.apply_gradients()` | Builds and trains the DQN policy network |

### Practical conventions

- Use channel-last tensors shaped like `(height, width, channels)` for TensorFlow.
- Keep frame processing fast: grayscale first, then resize to `84 x 84`.
- Release stale controller inputs before applying the next action.
- Treat reward signals as carefully tuned feedback, not as an afterthought.

---

## Step 1: Vision Pipeline and Temporal Frame Stacking

### Objective
Capture the game window, compress it, and preserve motion by stacking the last four processed frames.

### Why this matters
A single frame is a snapshot. A stacked tensor is a short memory. That short memory is what lets the agent infer direction, velocity, and attack timing.

```text
[Frame t-3]   [Frame t-2]   [Frame t-1]   [Frame t]
   84x84          84x84          84x84        84x84
                                         /
         +------------+------------+----------+
                           |
                           v
                    Stacked State Tensor
                        (84, 84, 4)
```

### Requirements
1. Enforce a fixed sampling interval so the model sees time consistently.
2. Keep a `deque(maxlen=4)` so the oldest frame drops automatically.
3. Stack the queue with `np.stack(..., axis=-1)` to form the model input tensor.

### Clean implementation sequence
- Capture raw pixels from the window bounds.
- Convert BGRA to grayscale.
- Resize to `84 x 84`.
- Append to the frame queue.
- Build the state tensor from the latest four frames.



---

## Step 2: Telemetry and Reward Engineering

### Core idea
Because the game does not expose internal state directly, reward must be inferred from visual cues.

### Player health tracking
- Crop the UI region where health masks appear.
- Threshold the crop for bright pixels.
- Compare the current white-pixel count to the previous step.
- Treat a significant drop as damage taken.

### Boss-hit detection
- Watch for sudden bright particle bursts around the combat zone.
- Use repeated identical frames as a freeze-frame hint for landed hits.
- Combine both signals for a more stable offensive reward.

### Reward matrix

| Telemetry trigger | Reward | Purpose |
| --- | --- | --- |
| Mask broken (player hit) | `-50.0` | Strongly penalizes failed dodges |
| Boss particle burst detected | `+20.0` | Rewards successful offense |
| Step survival incentive | `+0.1` | Encourages steady movement and survival |
| Stagnation penalty | `-0.05` | Prevents passive turtling |

### Design note
The reward scale should discourage standing still more than it encourages reckless aggression.

---

## Step 3: Actuator Mapping

### Action space
Each network output is an integer index that maps to a concrete controller behavior.

```python
ACTION_SPACE = {
    0: "NEUTRAL",
    1: "MOVE_LEFT",
    2: "MOVE_RIGHT",
    3: "JUMP",
    4: "ATTACK",
    5: "DASH",
    6: "MOVE_LEFT_AND_ATTACK",
    7: "MOVE_RIGHT_AND_ATTACK",
}
```

### Unlatching requirement
Before applying a new action, release the buttons from the previous action so inputs do not remain stuck across steps.

### Execution pattern
1. Clear the previous input state.
2. Choose the next action index.
3. Apply the corresponding joystick or button command.
4. Call `gamepad.update()` once to flush the entire state.

---

In [ ]:
ACTION_SPACE = {
    0: "NEUTRAL",
    1: "MOVE_LEFT",
    2: "MOVE_RIGHT",
    3: "JUMP",
    4: "ATTACK",
    5: "DASH",
    6: "MOVE_LEFT_AND_ATTACK",
    7: "MOVE_RIGHT_AND_ATTACK",
}

def reset_controller(gamepad):
    """Release stale inputs before the next action is applied."""
    gamepad.left_joystick_float(x_value_float=0.0, y_value_float=0.0)
    gamepad.release_button(vgamepad.XUSB_BUTTON.XUSB_GAMEPAD_A)
    gamepad.release_button(vgamepad.XUSB_BUTTON.XUSB_GAMEPAD_X)
    gamepad.release_button(vgamepad.XUSB_BUTTON.XUSB_GAMEPAD_RIGHT_SHOULDER)


## Step 4: The Network Brain and Gradient Engine

### Model structure
Your DQN should process the stacked `(84, 84, 4)` state tensor through a compact CNN pipeline:

1. `Conv2D` for broad spatial features.
2. `Conv2D` for tighter movement cues.
3. `Flatten` to collapse the feature maps.
4. `Dense(512, relu)` for nonlinear policy reasoning.
5. Linear output layer sized to the action space.

### Stabilization strategy
- The policy network learns every step.
- The target network stays frozen between syncs.
- Copy weights periodically, such as every 1,000 steps.

### Optimization loop
Wrap loss computation inside `tf.GradientTape()`, compute the Bellman target, and apply gradients with `optimizer.apply_gradients(...)`.

```text
policy_net -> Q-values -> action choice
replay buffer -> batch sampling -> Bellman target
target_net -> frozen bootstrap values
gradient tape -> loss -> optimizer step
```

---

In [ ]:
import tensorflow as tf

class DQN(tf.keras.Model):
    def __init__(self, action_count):
        super().__init__()
        self.conv1 = tf.keras.layers.Conv2D(32, 8, strides=4, activation="relu")
        self.conv2 = tf.keras.layers.Conv2D(64, 4, strides=2, activation="relu")
        self.flatten = tf.keras.layers.Flatten()
        self.hidden = tf.keras.layers.Dense(512, activation="relu")
        self.output_layer = tf.keras.layers.Dense(action_count, activation=None)

    def call(self, inputs, training=False):
        x = self.conv1(inputs)
        x = self.conv2(x)
        x = self.flatten(x)
        x = self.hidden(x)
        return self.output_layer(x)


## Step 5: The Master Optimization Loop

### Runtime blueprint
1. Initialize the window handle, screen capture, virtual controller, policy network, target network, and replay buffer.
2. Pause briefly so the game window can receive focus.
3. Enter the real-time loop and repeat the perception, telemetry, action, and learning cycle.
4. Sync the target network on a fixed schedule.
5. On exit, release controller inputs and shut everything down cleanly.

```text
while running:
    capture frame
    compute telemetry
    stack frames
    pick action
    send controller input
    store transition
    learn from replay
    sync target network periodically
```

### Closing note
> Start by validating the telemetry hook first. If the pixel-density signal is wrong, every later layer of the system will train on noise.

---

### Build order that keeps risk low
- Get the health-mask counter working first.
- Add frame preprocessing and the 4-frame stack next.
- Wire action translation only after telemetry is stable.
- Add the DQN and replay buffer last.
